<a href="https://colab.research.google.com/github/RodolfoFerro/modulo-deep-learning/blob/main/notebooks/06_Material_complementario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Aprendizaje profundo - Material complementario: Sesión 6  🧠**

> **Descripción:** Cuaderno complementario de contenidos del módulo de aprendizaje profundo para el Dimplomado en Ciencia de Datos de la ENES UNAM León, 2024-2026. <br>
> **Autor:** [Rodolfo Ferro](https://github.com/RodolfoFerro) <br>
> **Contacto:** [ferro@cimat.mx](mailto:ferro@cimat.mx)


## Contenido

### Sección I

1. Descarga de datos:
    - Usando `wget`
    - Conectando Google Drive a Colab
    - Con la API de Kaggle
2. Creación de un dataset de imágenes con `tf.data.Dataset`

### Sección II

3. Entrenamiento de una red neuronal convolucional (CNN)
4. Exportación de un modelo entrenado

## **Sección I**

### **Descarga de datos**

#### Usando `wget`

Este es el caso más clásico y fácil de entender. Basta con especificar el URL del archivo fuente y proceder con la descarga. De hecho, es un método que ya utilizamos previamente.

El comando se ejecuta desde lìnea de comandos, por lo que será necesario utilizar el símbolo `!`.

In [ ]:
!wget https://raw.githubusercontent.com/RodolfoFerro/apple-orange-dataset/main/training_data.csv

In [ ]:
import plotly.express as px
import pandas as pd


training_df = pd.read_csv('training_data.csv')
training_df['class_str'] = training_df['class'].astype('str')
training_df['hover'] = [text.split('/')[-1] for text in training_df['filename']]

fig = px.scatter_3d(
    training_df,
    x='r', y='g', z='b',
    color='class_str',
    symbol='class_str',
    color_discrete_sequence=['#be0900', '#ffb447'],
    opacity=0.5,
    hover_data=['hover']
)
fig.show()

#### Conectando Google Drive a Colab

Este es el otro caso fácil de entender. Basta con montar Drive al notebook.

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
!ls "/gdrive/MyDrive/Colab Notebooks"

#### Con la API de Kaggle

Este es un método que también resulta útil, aunque puede parecer u poco más elaborado.

In [ ]:
# Instalar Kaggle API
!pip install -q kaggle

Verificamos si existe el archivo de configuración para Kaggle, si no, lo creamos.

In [ ]:
!cat ~/.kaggle/kaggle.json

In [ ]:
!mkdir -p ~/.kaggle && \
echo '{"username":"rodolfoferro","key":"KGAT_622b50bfb7f477281fa7abec684cc57e"}' > ~/.kaggle/kaggle.json && \
chmod 600 ~/.kaggle/kaggle.json

Descargamos los datos con la información de la ruta del dataset:

In [ ]:
# Descargar dataset (requiere kaggle.json configurado)
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset

Para descomprimir:

In [ ]:
!mkdir -p data/images
!unzip microsoft-catsvsdogs-dataset.zip -d data/images

### **Creación de un dataset de TensorFlow**

Esta sección muestra cómo cargar un dataset de imágenes almacenado en disco utilizando
`tf.keras.utils.image_dataset_from_directory`.

Se asume la siguiente estructura de carpetas:

```
data/images/PetImages/
├── class_0/
│   ├── img001.jpg
│   └── ...
└── class_1/
    ├── img001.jpg
    └── ...
```

In [ ]:
import os
from PIL import Image

ROOT_DIR = "data/images/PetImages"

removed = []

for cls in os.listdir(ROOT_DIR):
    cls_path = os.path.join(ROOT_DIR, cls)
    if not os.path.isdir(cls_path):
        continue

    for fname in os.listdir(cls_path):
        fpath = os.path.join(cls_path, fname)

        # Eliminar archivos vacíos
        if os.path.getsize(fpath) == 0:
            os.remove(fpath)
            removed.append((fpath, "empty"))
            continue

        # Eliminar archivos no-imagen o corruptos
        try:
            with Image.open(fpath) as img:
                img.verify()
        except Exception:
            os.remove(fpath)
            removed.append((fpath, "corrupt"))

print(f"[INFO] Archivos eliminados: {len(removed)}")
for f, reason in removed[:10]:
    print(f" - {f} ({reason})")


In [ ]:
from PIL import Image
import os

SRC_ROOT = "data/images/PetImages"
DST_ROOT = "data/images_sanitized"

os.makedirs(DST_ROOT, exist_ok=True)

removed = 0
converted = 0

for cls in os.listdir(SRC_ROOT):
    src_cls = os.path.join(SRC_ROOT, cls)
    dst_cls = os.path.join(DST_ROOT, cls)

    if not os.path.isdir(src_cls):
        continue

    os.makedirs(dst_cls, exist_ok=True)

    for fname in os.listdir(src_cls):
        src_path = os.path.join(src_cls, fname)
        dst_path = os.path.join(dst_cls, fname)

        try:
            with Image.open(src_path) as img:
                img = img.convert("RGB")
                img.save(dst_path, format="JPEG", quality=95)
                converted += 1
        except Exception:
            removed += 1

print(f"[INFO] Imágenes convertidas: {converted}")
print(f"[INFO] Archivos eliminados: {removed}")


In [ ]:
import tensorflow as tf
import numpy as np


# Parámetros generales
DATA_DIR = "data/images_sanitized"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VALIDATION_SPLIT = 0.2

# Carga del dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
)

class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10), dpi=300)
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

Algo interesante de `tf.keras` es la posibilidad de crear un conjunto de trnasformaciones como si fueran un modelo de capas.

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]

# Validamos la transformación de pixeles a [0, 1]
print(np.min(first_image), np.max(first_image))

## **Sección II**

### **Entrenamiento de una CNN**

Podemos crear un modelo similar al que ya hemos creado anteriormente.

In [ ]:
import tensorflow as tf


# Parámetros
IMAGE_SIZE = (224, 224)
NUM_CLASSES = 2   # Clasificación multiclase
EPOCHS = 10
LEARNING_RATE = 1e-6


# Definición del modelo
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.summary()


Compilamos y entrenamos el modelo.

In [ ]:
# Compilación
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)


# Entrenamiento
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)

In [ ]:
import plotly.graph_objects as go


seen = "accuracy"

hist_values = history.history[seen]
hist_values_val = history.history['val_' + seen]
eje_x = np.arange(len(hist_values))

fig = go.Figure()

fig.add_trace(go.Scatter(x=eje_x,
                         y=hist_values,
                         mode='lines',
                         name=seen))
fig.add_trace(go.Scatter(x=eje_x,
                         y=hist_values_val,
                         mode='lines',
                         name='val_' + seen))
fig.update_layout(title='Historia de entrenamiento',
                   xaxis_title='Épocas',
                   yaxis_title=seen.capitalize())
fig.show()

### **Exportación de un modelo entrenado**

Exportamos el modelo entrenado como sigue.

In [ ]:
MODEL_PATH = "models/cnn_binary_classifier.keras"

# Crear carpeta si no existe
import os
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

# Guardar modelo completo
model.save(MODEL_PATH)

print(f"[INFO] Modelo almacenado en: {MODEL_PATH}")

Y para cargar un modelo guardado:

In [ ]:
import tensorflow as tf

MODEL_PATH = "models/cnn_binary_classifier.keras"

loaded_model = tf.keras.models.load_model(MODEL_PATH)
loaded_model.summary()

Podemos verificar que el modelo se importó correctamente.

In [ ]:
loaded_model.evaluate(val_ds)

In [ ]:
import numpy as np

for images, labels in val_ds.take(1):
    preds = loaded_model.predict(images)
    print(preds[:5])


--------

> Contenido creado por **Rodolfo Ferro**, 2024-2026. <br>
> Para cualquier comentario, puedes contactarme a través del correo [ferro@cimat.mx](mailto:ferro@cimat.mx).